# 11.3 — Partitioning and File Size

**Chapter 11, §3.3, layout decisions 1 and 2 (lines 236–238).**

The chapter says that the physical organization established *when the data is written* decides
how much a later query has to read, and that partitioning on a column with too many distinct
values trades that saving for "large numbers of small partitions and excessive metadata
overhead".

Both halves are measurable. One table is written four ways, one query is run against each, and
the files the scan actually opens are counted.

Chapter 5 owns partition sizing inside a job and chapter 3 owns Parquet's own pruning. This
notebook is about the directory layout on disk and nothing else.

In [1]:
import os, shutil, tempfile, time
from pyspark.sql import SparkSession, functions as F

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch11-layout")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH)

spark = (
    SparkSession.builder.appName("CS777-11.3")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", 8)
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.version

'4.2.0'

In [2]:
# 200,000 synthetic orders. Everything is derived from the row number, so the table is
# identical on every run and on every machine.
COUNTRIES = ["US", "DE", "FR", "JP", "BR"]

orders = (
    spark.range(0, 200_000)
    .withColumn("day", F.date_add(F.lit("2026-03-01").cast("date"), (F.col("id") % 10).cast("int")))
    .withColumn("country", F.element_at(F.array(*[F.lit(c) for c in COUNTRIES]),
                                        ((F.col("id") / 10).cast("int") % 5 + 1).cast("int")))
    .withColumn("user_id", (F.col("id") % 2000).cast("int"))
    .withColumn("amount", (F.col("id") % 97).cast("double"))
    .select("id", "country", "day", "user_id", "amount")
)
orders.show(5)
print("rows:", orders.count())

+---+-------+----------+-------+------+
| id|country|       day|user_id|amount|
+---+-------+----------+-------+------+
|  0|     US|2026-03-01|      0|   0.0|
|  1|     US|2026-03-02|      1|   1.0|
|  2|     US|2026-03-03|      2|   2.0|
|  3|     US|2026-03-04|      3|   3.0|
|  4|     US|2026-03-05|      4|   4.0|
+---+-------+----------+-------+------+
only showing top 5 rows
rows: 200000


## 1. Four layouts of the same table

`flat` is one directory of Parquet files. The other three push one or two columns into the
directory names, which is the "directory partitioning" of §3.3.

In [3]:
LAYOUTS = {
    "flat":              ([], False),
    "by country":        (["country"], False),
    "by country, day":   (["country", "day"], False),
    "same, regrouped":   (["country", "day"], True),   # one file per directory
    "by user_id":        (["user_id"], False),
}
paths = {}
for label, (cols, regroup) in LAYOUTS.items():
    path = os.path.join(SCRATCH, label.strip().replace(", ", "_").replace(" ", "-"))
    df = orders.repartition(*cols) if regroup else orders
    writer = df.write.mode("overwrite")
    if cols:
        writer = writer.partitionBy(*cols)
    writer.parquet(path)
    paths[label] = path

top = paths["by country, day"]
for entry in sorted(d for d in os.listdir(top) if os.path.isdir(os.path.join(top, d)))[:2]:
    inner = sorted(os.listdir(os.path.join(top, entry)))
    print(entry + "/", inner[:3], "...", len(inner), "sub-directories")

country=BR/ ['day=2026-03-01', 'day=2026-03-02', 'day=2026-03-03'] ... 10 sub-directories
country=DE/ ['day=2026-03-01', 'day=2026-03-02', 'day=2026-03-03'] ... 10 sub-directories


The partition column is gone from the files and lives in the path. `country=DE/day=2026-03-01/`
is the chapter's `year=2025/month=07/`, and a reader recovers the column by parsing the
directory name.

## 2. What a filtered query reads

The instrument is the one the SQL tab of the Spark UI shows on the scan node. It is worth being
precise about what it can see: **`size of files read` is the size of the file splits assigned to
tasks**, not the bytes fetched, so it is blind to column pruning inside a file. The file *count*
is the number this notebook relies on.

In [4]:
SIZE_UNITS = {"B": 1, "KiB": 1024, "MiB": 1024**2, "GiB": 1024**3, "TiB": 1024**4}


def spark_scan_metrics():
    # `number of files read` / `size of files read` for the query that just ran --
    # the same numbers the SQL tab of the Spark UI shows on the scan node.
    store = spark._jsparkSession.sharedState().statusStore()
    execs = store.executionsList()
    last = execs.apply(execs.size() - 1)
    values, planned = store.executionMetrics(last.executionId()), last.metrics()
    seen = {}                                    # dedupe: AQE republishes the same accumulators
    for i in range(planned.size()):
        m = planned.apply(i)
        v = values.get(m.accumulatorId())
        if v.isDefined():
            seen[m.accumulatorId()] = (m.name(), v.get())
    files, split_bytes = 0, 0.0
    for name, text in seen.values():
        line = text.strip().split("\n")[-1].split("(")[0].strip()
        if name == "number of files read":
            files += int(line.replace(",", ""))
        elif name == "size of files read":
            num, unit = line.split()
            split_bytes += float(num.replace(",", "")) * SIZE_UNITS[unit]
    return {"files_read": files, "split_KB": round(split_bytes / 1024, 1)}

In [5]:
# One question, asked of every layout: what did German customers spend on 3 March?
def query(path):
    return (spark.read.parquet(path)
            .filter((F.col("country") == "DE") & (F.col("day") == "2026-03-03"))
            .agg(F.sum("amount")))


print(f"{'layout':<17} {'answer':>10} {'on disk':>8} {'files read':>11} {'split KB':>9}")
for label, path in paths.items():
    answer = query(path).collect()[0][0]
    on_disk = sum(len([f for f in fs if f.endswith(".parquet")])
                  for _, _, fs in os.walk(path))
    m = spark_scan_metrics()
    print(f"{label:<17} {answer:>10,.0f} {on_disk:>8} {m['files_read']:>11} {m['split_KB']:>9.1f}")

layout                answer  on disk  files read  split KB


flat                 192,085       18          18    1056.2
by country           192,085       90          18     252.4


by country, day      192,085      900          18      49.2
same, regrouped      192,085       50           1      19.1


by user_id           192,085    36000       36000   46080.0


Every layout returns the same answer, and the bytes assigned to tasks fall by a factor of five
and then fifty as the filter columns move into the directory names. That elimination is
*partition pruning*.

The **file count does not fall with it**, and the reason is the subject of §4 below: eighteen
writing tasks each held rows for every country and every day, so each of the fifty directories
got eighteen files and the DE/3-March directory is eighteen files rather than one. Regrouping the
rows before the write — the `same, regrouped` row — puts one file in each directory, and only then
does the scan open a single file. Partitioning decides *which* directories are read; it does
nothing about how many files are in them.

The optimizer records the pruning explicitly in the plan.

In [6]:
import re

for label in ("flat", "by country", "by country, day"):
    plan = query(paths[label])._jdf.queryExecution().executedPlan().toString()
    found = re.search(r"PartitionFilters: \[[^\]]*\]", plan).group(0)
    print(f"{label:>16}: {found[:100]}")

            flat: PartitionFilters: []
      by country: PartitionFilters: [isnotnull(country#134), (country#134 = DE)]
 by country, day: PartitionFilters: [isnotnull(country#148), isnotnull(day#149), (country#148 = DE), (day#149 = 2026-0


An empty `PartitionFilters` against one that carries the predicate. A filter that reaches
`PartitionFilters` is answered from directory names during planning; the same predicate on the
flat table lands in `DataFilters` instead and is applied after the rows have been read.

## 3. The cost side: what `by user_id` bought

Partitioning on a column with two thousand distinct values would prune just as well *for a query
that filtered on `user_id`*. The question is what the table looks like afterwards — and what it
costs a query that filters on something else, as the one above does.

In [7]:
def files_of(path):
    return [os.path.join(r, f) for r, _, fs in os.walk(path) for f in fs if f.endswith(".parquet")]


print(f"{'layout':<16} {'directories':>12} {'files':>7} {'mean KB':>9} {'total MB':>9}")
for label, path in paths.items():
    fs = files_of(path)
    dirs = sum(1 for r, ds, _ in os.walk(path) for d in ds)
    total = sum(os.path.getsize(f) for f in fs)
    print(f"{label:<16} {dirs:>12} {len(fs):>7} "
          f"{total / len(fs) / 1024:>9.1f} {total / 1024**2:>9.1f}")

layout            directories   files   mean KB  total MB
flat                        0      18      58.7       1.0
by country                  5      90      14.0       1.2
by country, day            55     900       2.7       2.4
same, regrouped            55      50      19.1       0.9


by user_id               2000   36000       1.3      45.0


Two thousand directories, and the *same* two hundred thousand rows now occupy substantially more
space than the flat layout: each tiny file carries its own Parquet footer, its own schema, and its
own column statistics, and each one is too small for the encoders to find much to compress.

The listing cost is the other half. 11.1 measured it in a bucket; here it is visible as planning
time on a query that needs the whole table.

In [8]:
for label in ("flat", "by user_id"):
    t0 = time.time()
    n = spark.read.parquet(paths[label]).count()
    print(f"{label:<12} full scan of {n:,} rows: {time.time() - t0:>5.2f} s "
          f"over {len(files_of(paths[label])):,} files")

flat         full scan of 200,000 rows:  0.07 s over 18 files


by user_id   full scan of 200,000 rows:  5.38 s over 36,000 files


## 4. Controlling how many files a write produces

The number of output files is the number of tasks that write, multiplied by the number of
partition values each task happens to hold. Two levers are named in §3.3.

In [9]:
out = os.path.join(SCRATCH, "sizing-a")
(orders.repartition(8).write.mode("overwrite")       # eight writing tasks
       .partitionBy("country").parquet(out))
print(f"{'8 tasks x 5 countries':<44} {len(files_of(out)):>4} files")

out = os.path.join(SCRATCH, "sizing-b")
(orders.repartition("country").write.mode("overwrite")
       .partitionBy("country").parquet(out))
print(f"{'repartitioned by country before the write':<44} {len(files_of(out)):>4} files")

out = os.path.join(SCRATCH, "sizing-c")
(orders.repartition(1).write.mode("overwrite")
       .option("maxRecordsPerFile", 50_000).parquet(out))
print(f"{'one task, maxRecordsPerFile = 50,000':<44} {len(files_of(out)):>4} files")

8 tasks x 5 countries                          40 files
repartitioned by country before the write       5 files


one task, maxRecordsPerFile = 50,000            4 files


Eight tasks each holding rows for all five countries write forty files; shuffling on the
partition column first sends each country to one task and writes five. `maxRecordsPerFile` works
from the other end, splitting a single task's output at a row count rather than merging it.

This is also the arithmetic Exercise 4(b) asks for. Writing a table bucketed into 96 buckets from
200 uncoalesced tasks can leave $96 \times 200 = 19{,}200$ files, for the same reason the first
line above leaves 40: every task contributes at most one file to every bucket it holds a row for.

## Conclusion

* Partitioning on the columns a query filters on lets the planner discard directories before
  reading anything: 1 file opened instead of every file in the table.
* The saving is bounded by how many distinct values the column has. At two thousand values the
  table becomes thousands of files that are individually too small to compress well and
  collectively expensive to list.
* File count at write time is a product of tasks and partition values, and both `repartition`
  and `maxRecordsPerFile` exist to control it.

§3.3's closing sentence is the sequel: a table format that keeps per-file column statistics in
metadata can skip files on columns that were never used for partitioning at all, which is what
11.6 goes on to show.

In [10]:
spark.stop()